# Séries temporelles et LSTM — ce que le temps apporte, et ce qu'il n'apporte pas

Le modèle du projet répond à **où** : quelles communes surveiller aujourd'hui.
C'est une classification sur un panneau de 34 734 séries simultanées, dont 80 %
ne contiennent aucun événement.

Ce notebook explore l'axe que le modèle principal laisse de côté — le **temps** —
et répond à trois questions qui reviennent systématiquement :

| Question | Outil | Verdict |
|---|---|---|
| La série est-elle stationnaire ? | **ADF** | oui, et ce n'est pas une bonne nouvelle |
| Quel ordre pour un modèle autorégressif ? | **ACF / PACF** | AR(2), et seulement sur les *résidus* du cycle |
| Combien de feux demain en France ? | **SARIMAX** | −21,5 % de MAE sur la référence saisonnière |
| Un LSTM ferait-il mieux ? | **bootstrap apparié** | non — **−23,6 %**, IC [−33,5 ; −17,3] |

La dernière ligne est la conclusion principale, et elle demande à être justifiée
plutôt qu'assénée : un LSTM correctement optimisé perd contre un gradient
boosting qui voit vingt fois moins d'historique météo. La section 6 explique
pourquoi ce n'est pas un accident.

> **Reproduire les chiffres** — `python -m tvfed.series` puis
> `python -m tvfed.lstm --essais 25`, `--final`, et enfin `python -m tvfed.comparer`.

In [ ]:
# ── enregistrement automatique des figures ──
# chaque plt.show() écrit aussi un PNG dans figures/series-lstm/
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from tvfed.figures import activer

activer("series-lstm")

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

for p in (Path.cwd(), *Path.cwd().parents):
    if (p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(p / "src"))
        RACINE = p
        break

from tvfed import db
from tvfed import series as SER

# charte identique aux notebooks d'audit
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLEU, ORANGE, ROUGE, VERT, VIOLET = "#2a78d6", "#eb6834", "#e34948", "#1baf7a", "#4a3aa7"
GRIS = "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "font.size": 9, "axes.edgecolor": "#c3c2b7", "text.color": INK})

PROC = RACINE / "data" / "processed"

D = SER.serie()                       # feux/jour et FWI national, 2006-2025
ADF = pd.read_csv(PROC / "series_adf.csv")
SARIMAX_R = pd.read_csv(PROC / "series_sarimax.csv")
CMP = pd.read_csv(PROC / "comparaison_appariee.csv")
PRAUC = pd.read_csv(PROC / "pr_auc_val.csv")
PARAMS = json.loads((PROC / "best_params_lstm.json").read_text(encoding="utf-8"))

print(f"{len(D):,} jours (2006-2025) · {D.feux.sum():,} communes-jours en feu")
print(f"  moyenne {D.feux.mean():.2f}/jour · médiane {D.feux.median():.0f} · "
      f"max {D.feux.max()} le {D.feux.idxmax().date()}")
print(f"  {(D.feux == 0).mean():.1%} des jours sans aucun feu en France")

## 1. La série : très saisonnière, très asymétrique

Avant tout test, il faut regarder la série. Deux traits la caractérisent, et
tous deux contraignent le choix de modèle.

**Elle est fortement asymétrique à droite.** La médiane est à 4 communes en feu
par jour, mais la queue monte jusqu'à 89. La moyenne (6,7) tombe entre les deux
et ne décrit ni l'un ni l'autre — c'est ce qui justifiera de regarder la MAE
plutôt que le RMSE en section 4, le second étant dominé par une poignée de
journées d'août.

**Le cycle annuel domine tout.** C'est la raison pour laquelle l'ACF brute de
la section 3 sera inexploitable : elle ne mesurera que « c'est l'été ».

In [ ]:
'''FIG 1 — La série nationale : le cycle, une saison, et la distribution.

(a) rend visible que le signal est un empilement de saisons, pas une série
    stationnaire ordinaire ;
(b) montre à quoi ressemble UNE saison — un fond d'été plus quelques pics
    isolés qui font l'essentiel du total ;
(c) explique pourquoi la MAE est la bonne métrique : la distribution est si
    asymétrique qu'un RMSE ne parlerait que des trois pires jours.
'''
fig = plt.figure(figsize=(15.5, 6.4))
gs = fig.add_gridspec(2, 2, width_ratios=[2.15, 1], height_ratios=[1, 1],
                      hspace=.52, wspace=.24)

# ── (a) 20 ans ───────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, :])
ax.fill_between(D.index, 0, D.feux, color=ORANGE, lw=0, alpha=.85)
ax.set_xlim(D.index[0], D.index[-1])
ax.set_ylabel("communes en feu")
ax.set_title("(a)  Vingt ans de départs de feu, France entière",
             fontsize=11.5, weight="bold", loc="left")
pic = D.feux.idxmax()
ax.annotate(f"{D.feux.max()} communes\nle {pic.strftime('%d/%m/%Y')}",
            xy=(pic, D.feux.max()), xytext=(-90, -14),
            textcoords="offset points", fontsize=8.5, color=MUTED,
            arrowprops=dict(arrowstyle="-", color=GRIS, lw=.9))

# ── (b) une saison ───────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
an = D[D.index.year == 2022]
ax.fill_between(an.index, 0, an.feux, color=ROUGE, lw=0, alpha=.8)
ax.set_xlim(an.index[0], an.index[-1])
ax.set_title("(b)  2022 dans le détail", fontsize=10.5, weight="bold",
             loc="left")
ax.set_ylabel("communes en feu")

# ── (c) distribution ─────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
v = D.feux.to_numpy()
ax.hist(v, bins=np.arange(0, v.max() + 5, 4), color=BLEU, alpha=.85, lw=0)
ax.set_yscale("log")
ax.set_xlabel("communes en feu ce jour-là")
ax.set_ylabel("nombre de jours")
ax.set_title("(c)  Une distribution écrasée sur zéro", fontsize=10.5,
             weight="bold", loc="left")
q = np.percentile(v, [50, 95, 99])
for s, lab in zip(q, ["médiane", "95ᵉ", "99ᵉ"]):
    ax.axvline(s, color=MUTED, ls=":", lw=1)
    ax.text(s, ax.get_ylim()[1] * .55, f" {lab}\n {s:.0f}", fontsize=7.5,
            color=MUTED)

for a in fig.axes:
    a.grid(color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
plt.show()

print(f"les 1 % de jours les plus actifs concentrent "
      f"{v[v >= q[2]].sum() / v.sum():.1%} de tous les départs de feu")

## 2. Stationnarité — le test de Dickey-Fuller augmenté

> **H0 : la série a une racine unitaire, c'est-à-dire qu'elle n'est PAS
> stationnaire.**
>
> Rejeter H0 (p < 0,05) signifie donc **stationnaire**. C'est l'inverse de
> l'intuition, et la confusion la plus fréquente sur ce test.

Une série stationnaire a une moyenne et une variance qui ne dérivent pas. C'est
la condition d'application d'ARIMA : le modèle suppose que les coefficients
appris hier valent encore demain.

In [ ]:
print("H0 : racine unitaire (NON stationnaire).  p < 0,05 → STATIONNAIRE\n")
print(f"{'série':32s} {'stat ADF':>9s} {'p':>11s} {'retards':>8s}  conclusion")
print("─" * 78)
for _, r in ADF.iterrows():
    print(f"{r.serie:32s} {r.adf:9.2f} {r.p:11.2e} {r.retards:8d}  "
          f"{'STATIONNAIRE' if r.stationnaire else 'non stationnaire'}")

### Ce que ce tableau dit vraiment

Les quatre séries sont stationnaires, **y compris le total annuel sur 20 points**.

C'est un résultat gênant, et il vaut mieux le dire tout de suite : *sur la
fenêtre 2006-2025, le nombre de feux ne montre aucune tendance.* La régression
linéaire sur les totaux annuels donne p = 0,89 — rigoureusement rien.

Ça n'invalide pas le projet, mais ça déplace la charge de la preuve. Si les feux
ne montent pas, sur quoi repose une projection à 2050 ? La réponse est en
section 5 : **la tendance est dans le FWI, pas dans les feux** — et il faut
53 ans de données pour la voir, pas 20.

Deuxième lecture : la série différenciée est *elle aussi* stationnaire, et
plus fortement (ADF −24,6). C'est le signe qu'il ne faut **pas** différencier —
le paramètre `d` d'ARIMA restera à 0. Différencier une série déjà stationnaire
ajoute du bruit sans rien retirer.

## 3. ACF et PACF — choisir l'ordre, mais sur la bonne série

L'**ACF** mesure la corrélation entre $y_t$ et $y_{t-k}$, tout ce qui les relie
compris — y compris ce qui passe par les jours intermédiaires. Elle indique
l'ordre **MA**.

La **PACF** mesure ce qui reste de cette corrélation **une fois retiré l'effet
des retards intermédiaires**. Elle indique l'ordre **AR** : c'est le retard
auquel elle décroche qui donne $p$.

⚠️ **Sur la série brute, les deux sont inutilisables.** Le cycle annuel domine à
un point tel que tous les retards paraissent corrélés — non parce que le 3 août
informe sur le 4 août, mais parce que les deux sont en été. Il faut d'abord
retirer la saison, ici par régression sur quatre harmoniques de Fourier, puis
lire l'ACF/PACF **des résidus**.

In [ ]:
'''FIG 2 — ACF et PACF, avant et après retrait du cycle annuel.

La ligne du haut est un piège classique : l'ACF brute décroît lentement et
ressemble à une série non stationnaire à mémoire longue. Ce n'est qu'un cycle.
La ligne du bas montre la vraie structure — courte, deux retards.
'''
from statsmodels.api import OLS, add_constant
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

H = SER.harmoniques(D.index)
RESID = pd.Series(OLS(D.feux.to_numpy(), add_constant(H.to_numpy())).fit().resid,
                  index=D.index)

fig, ax = plt.subplots(2, 2, figsize=(15.5, 6.2))
plot_acf(D.feux, lags=60, ax=ax[0, 0], color=BLEU, vlines_kwargs={"colors": BLEU})
plot_pacf(D.feux, lags=60, ax=ax[0, 1], method="ywm", color=BLEU,
          vlines_kwargs={"colors": BLEU})
plot_acf(RESID, lags=60, ax=ax[1, 0], color=ORANGE, vlines_kwargs={"colors": ORANGE})
plot_pacf(RESID, lags=60, ax=ax[1, 1], method="ywm", color=ORANGE,
          vlines_kwargs={"colors": ORANGE})
for a, t in zip(ax.ravel(),
                ["ACF — série brute  (illisible : le cycle annuel écrase tout)",
                 "PACF — série brute",
                 "ACF — après retrait du cycle annuel",
                 "PACF — après retrait du cycle  ← c'est celle-ci qui donne p"]):
    a.set_title(t, fontsize=10, weight="bold", loc="left")
    a.grid(color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
    a.set_xlabel("retard (jours)")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import pacf as pacf_f

p_ = pacf_f(RESID.dropna(), nlags=12, method="ywm")
seuil = 1.96 / np.sqrt(len(RESID))
print(f"PACF des résidus — seuil de significativité ±{seuil:.4f}\n")
for k in range(1, 11):
    barre = "█" * int(abs(p_[k]) / .02)
    print(f"  retard {k:>2}  {p_[k]:+.4f}  {barre:<22s}"
          f"{'significatif' if abs(p_[k]) > seuil else ''}")
signif = [k for k in range(1, 11) if abs(p_[k]) > seuil]
faible = min(k for k in range(1, 11) if abs(p_[k]) < .10)
print(f"\n→ dernier retard SIGNIFICATIF : {max(signif)}  "
      f"(PACF = {p_[max(signif)]:+.3f}, soit {p_[max(signif)] / p_[1]:.1%} "
      f"du retard 1)")
print(f"→ mais la PACF tombe sous 0,10 dès le retard {faible} : passé là, "
      f"« significatif » ne veut plus dire « utile »")
print("→ ordre retenu : AR(2)   ·   série stationnaire (§2) → d = 0")

### Lecture — et pourquoi « significatif » ne veut pas dire « utile »

Le test formel désigne le **retard 8** comme dernier retard significatif. Il
faut résister à la tentation de lire ce chiffre tel quel.

Le seuil vaut $1{,}96/\sqrt{7\,305} = \pm 0{,}023$ : avec 7 305 points, des
corrélations minuscules franchissent la barre. Or les valeurs réelles
s'effondrent :

| retard | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|
| PACF | **0,697** | **0,191** | 0,077 | 0,041 | 0,024 | 0,048 | 0,015 | 0,034 |

Chaque retard divise la corrélation par un facteur ~3 à 4 jusqu'au troisième,
puis tout flotte au ras du seuil. **Statistiquement détectable, pratiquement
négligeable.** C'est la différence entre significativité et taille d'effet, et
c'est exactement le genre de chiffre qu'un grand échantillon rend trompeur.

D'où **AR(2)** — c'est l'ordre retenu par `tvfed.series` — et `d = 0` puisque la
série est stationnaire.

Ce que ça dit sur le fond : **l'essentiel de l'autocorrélation est épuisé en
deux à trois jours.** Un feu aujourd'hui informe sur demain, un peu sur
après-demain, puis presque plus. Tout ce qui ressemblait à une structure au-delà
était le cycle saisonnier.

C'est le premier indice sérieux contre le LSTM. Une architecture conçue pour
capter des dépendances longues n'a pas grand-chose à capter ici — passé quelques
jours, ce qui reste est porté par la météo, pas par l'historique des feux.

### Pourquoi pas de composante SARIMA saisonnière

Une saisonnalité annuelle sur données journalières donnerait $s = 365$. Un
SARIMA$(p,d,q)(P,D,Q)_{365}$ demanderait d'estimer des coefficients à 365 pas de
distance sur 5 113 points : ingérable et instable. La pratique établie est de
porter la saisonnalité par des **termes de Fourier en exogène** — c'est le « X »
de SARIMAX qui travaille.

## 4. SARIMAX — répondre au « combien »

Le protocole reprend celui du modèle principal : **ajustement 2006-2019,
évaluation 2020-2022**. Le test 2023-2025 n'est pas touché.

> Une première version évaluait sur 2020-2025, ce qui recouvrait le test. Ce
> n'était pas faux au sens strict — cible différente, modèle différent, aucune
> décision prise dessus. Mais le risque d'un jeu de test est **cumulatif** : un
> projet qui s'autorise une exception « parce que ce n'est pas vraiment la même
> chose » finit par n'avoir plus de juge du tout. L'évaluation s'arrête donc
> en 2022.

Trois variantes, pour isoler ce que chaque composante apporte.

In [ ]:
'''FIG 3 — Ce que chaque composante de SARIMAX apporte.

(a) confronte les trois variantes à l'observé sur 2020-2022. L'ARIMA seul est
    plat : à 1 096 pas d'horizon, un AR(2) a oublié depuis longtemps son point
    de départ, et prédit la moyenne.
(b) chiffre l'écart en MAE, référence saisonnière comprise.
'''
from statsmodels.tsa.statespace.sarimax import SARIMAX

an = D.index.year
TR, TE = D[an <= 2019], D[(an >= 2020) & (an <= 2022)]
h_tr, h_te = SER.harmoniques(TR.index), SER.harmoniques(TE.index)
exo_tr = pd.concat([h_tr, TR[["fwi", "part_danger"]]], axis=1)
exo_te = pd.concat([h_te, TE[["fwi", "part_danger"]]], axis=1)

VARIANTES = {
    "Fourier + FWI": (exo_tr, exo_te, ROUGE),
    "Fourier seul": (h_tr, h_te, BLEU),
    "sans exogène": (None, None, GRIS),
}
pred = {}
for nom, (Xa, Xb, _) in VARIANTES.items():
    m = SARIMAX(TR.feux, exog=Xa, order=(2, 0, 1), enforce_stationarity=False,
                enforce_invertibility=False).fit(disp=False)
    pred[nom] = np.clip(m.get_forecast(steps=len(TE), exog=Xb).predicted_mean, 0, None)

clim = TR.groupby(TR.index.dayofyear).feux.mean()
naif = pd.Series(TE.index.dayofyear.map(clim).to_numpy(), index=TE.index)
mae = {n: float(np.abs(p - TE.feux).mean()) for n, p in pred.items()}
mae["référence saisonnière"] = float(np.abs(naif - TE.feux).mean())

fig = plt.figure(figsize=(15.5, 5.6))
gs = fig.add_gridspec(1, 2, width_ratios=[2.5, 1], wspace=.22)

ax = fig.add_subplot(gs[0])
ax.fill_between(TE.index, 0, TE.feux, color=GRIS, lw=0, alpha=.75,
                label="observé", zorder=1)
for nom, (_, _, coul) in VARIANTES.items():
    ax.plot(TE.index, pred[nom], color=coul, lw=1.35, label=nom, zorder=3)
ax.set_xlim(TE.index[0], TE.index[-1])
ax.set_ylabel("communes en feu")
ax.legend(frameon=False, ncols=4, fontsize=9, loc="upper left")
ax.set_title("(a)  Prévision 2020-2022, ajustée sur 2006-2019",
             fontsize=11.5, weight="bold", loc="left")

ax = fig.add_subplot(gs[1])
ordre = sorted(mae, key=mae.get)
coul = {"Fourier + FWI": ROUGE, "Fourier seul": BLEU, "sans exogène": GRIS,
        "référence saisonnière": VIOLET}
ax.barh(range(len(ordre)), [mae[n] for n in ordre],
        color=[coul[n] for n in ordre], height=.62)
ax.set_yticks(range(len(ordre)), ordre, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("MAE (communes/jour)")
ax.set_title("(b)  Erreur absolue moyenne", fontsize=11.5, weight="bold",
             loc="left")
for i, n in enumerate(ordre):
    ax.text(mae[n] + .12, i, f"{mae[n]:.2f}", va="center", fontsize=9,
            color=INK)
ax.set_xlim(0, max(mae.values()) * 1.22)

for a in fig.axes:
    a.grid(color=GRID, lw=.7, axis="both")
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
plt.show()

print(SARIMAX_R.to_string(index=False))
g = 100 * (1 - mae["Fourier + FWI"] / mae["référence saisonnière"])
print(f"\ngain du meilleur SARIMAX sur la référence saisonnière : {g:+.1f} % de MAE")

### Ce que les trois variantes démontrent

| Variante | MAE | r | Lecture |
|---|---|---|---|
| **Fourier + FWI** | **4,03** | 0,850 | la météo fait le travail |
| Fourier seul | 6,42 | 0,603 | le calendrier seul explique déjà beaucoup |
| ARIMA sans exogène | 8,34 | **−0,118** | **inutilisable** |

La troisième ligne mérite qu'on s'y arrête : la corrélation est **négative**. À
1 096 pas d'horizon, un AR(2) dont la mémoire utile est de deux jours a
totalement oublié son point de départ ; il converge vers la moyenne de la série
et la ligne plate qu'il produit est légèrement anti-corrélée à l'observé, par
hasard. **Ce n'est pas un mauvais réglage, c'est structurel** : aucun modèle
purement autorégressif ne peut prévoir à trois ans.

Ajouter le FWI fait tomber la MAE de 37 % par rapport au calendrier seul. La
prévisibilité du feu n'est pas dans son propre passé — elle est dans la météo.
C'est exactement ce que le modèle principal exploite, et c'est aussi
l'explication de la section 6.

## 5. La tendance de fond : 53 ans, pas 20

La section 2 a laissé une question ouverte : si le nombre de feux ne monte pas
sur 2006-2025, sur quoi repose une projection à 2050 ?

La réponse tient à la fenêtre d'observation. Vingt points annuels, sur une série
aussi bruitée, n'ont pas la puissance statistique de détecter une tendance
modérée. Le FWI, lui, est disponible depuis **1973** — 53 ans, et 21,9 millions
de lignes de météo.

In [ ]:
'''FIG 4 — La tendance est dans le FWI, pas dans le décompte des feux.

(a) FWI estival moyen, 1973-2025. La pente est nette et hautement significative.
(b) le même exercice sur les feux, 2006-2025 : rien. Deux fenêtres
    différentes, deux conclusions différentes — et c'est la plus longue qui
    porte l'information climatique.
'''
from scipy import stats

with db.connexion() as c:
    FWI_AN = pd.read_sql(
        "SELECT extract(year FROM date)::int AS an, avg(fwi) AS fwi "
        "FROM fait_meteo "
        "WHERE extract(month FROM date) BETWEEN 6 AND 9 "
        "GROUP BY 1 ORDER BY 1", c)

feux_an = D.feux.resample("YE").sum()
feux_an.index = feux_an.index.year

fig, ax = plt.subplots(1, 2, figsize=(15.5, 4.9), width_ratios=[1.35, 1])


def tendance(a, x, y, coul, titre, unite):
    r = stats.linregress(x, y)
    a.plot(x, y, "o-", color=coul, lw=1.15, ms=3.6, alpha=.85)
    a.plot(x, r.intercept + r.slope * np.asarray(x), color=INK, lw=1.6, ls="--")
    dep = 100 * r.slope * (x[-1] - x[0]) / (r.intercept + r.slope * x[0])
    a.set_title(titre, fontsize=11.5, weight="bold", loc="left")
    a.set_ylabel(unite)
    a.text(.025, .93, f"pente {r.slope:+.4f}/an   p = {r.pvalue:.1e}\n"
                      f"soit {dep:+.0f} % sur {x[-1] - x[0]} ans",
           transform=a.transAxes, va="top", fontsize=9.5,
           color=INK if r.pvalue < .05 else MUTED,
           bbox=dict(fc="#fcfcfb", ec=GRIS, lw=.8, pad=4.5))
    return r


r1 = tendance(ax[0], FWI_AN.an.to_numpy(), FWI_AN.fwi.to_numpy(), ORANGE,
              "(a)  FWI moyen juin-septembre, 1973-2025", "FWI")
r2 = tendance(ax[1], feux_an.index.to_numpy(), feux_an.to_numpy(), ROUGE,
              "(b)  Communes-jours en feu par an, 2006-2025", "communes-jours")

for a in ax:
    a.grid(color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
    a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.show()

print(f"FWI estival  : p = {r1.pvalue:.2e}   "
      f"{'SIGNIFICATIF' if r1.pvalue < .05 else 'non significatif'}")
print(f"feux par an  : p = {r2.pvalue:.2f}     "
      f"{'SIGNIFICATIF' if r2.pvalue < .05 else 'NON significatif'}")

# ⚠️ L'ampleur de la hausse DÉPEND de l'agrégation choisie. On le montre,
# plutôt que de laisser croire qu'il existe un chiffre unique.
with db.connexion() as c:
    AN_TOT = pd.read_sql(
        "SELECT extract(year FROM date)::int AS an, avg(fwi) AS fwi "
        "FROM fait_meteo GROUP BY 1 ORDER BY 1", c)
r3 = stats.linregress(AN_TOT.an, AN_TOT.fwi)
h3 = 100 * r3.slope * 52 / (r3.intercept + r3.slope * 1973)
print(f"\nselon l'agrégation retenue :")
print(f"  moyenne juin-septembre  {100 * r1.slope * 52 / (r1.intercept + r1.slope * 1973):+.0f} % "
      f"sur 52 ans   p = {r1.pvalue:.1e}")
print(f"  moyenne annuelle        {h3:+.0f} % sur 52 ans   p = {r3.pvalue:.1e}")
print("  → le notebook retient l'ESTIVALE : c'est la saison qui porte les feux")

### Deux fenêtres, deux conclusions — et il faut expliquer laquelle croire

Le **danger météorologique monte de façon incontestable** sur 53 ans. Le
**nombre de feux, lui, ne bouge pas** sur 20 ans.

> ⚠️ **Citer le chiffre avec sa définition.** La hausse du FWI dépend de
> l'agrégation : la moyenne **juin-septembre** monte plus que la moyenne
> annuelle, puisque c'est l'été qui se réchauffe et s'assèche le plus. Le
> notebook retient l'estivale — c'est la saison qui porte les feux — et affiche
> les deux, pour qu'aucun chiffre ne circule sans son périmètre.

Ce n'est pas une contradiction, et il serait malhonnête de ne montrer que le
premier graphique. Trois lectures cohabitent :

1. **La puissance statistique.** Vingt points annuels très bruités ne peuvent
   pas détecter une tendance modérée. L'absence de preuve n'est pas une preuve
   d'absence.
2. **La prévention fonctionne.** Le nombre de départs dépend autant des moyens
   de lutte, des interdictions d'accès aux massifs et du débroussaillement que
   du climat. Un aléa qui monte à sinistralité constante est le résultat attendu
   d'une politique de prévention efficace.
3. **Ce que le modèle projette, c'est l'aléa, pas le bilan.** Les projections à
   2050 du projet transportent l'évolution du **FWI** sous les scénarios RCP,
   pas une extrapolation du décompte de feux. C'est la seule des deux quantités
   qui montre un signal, et la seule qu'un modèle climatique sait fournir.

> **À dire en soutenance** : ne pas annoncer « les feux augmentent » — les
> données du projet ne le montrent pas. Annoncer « les conditions favorables aux
> feux augmentent de façon très significative, et le nombre de départs reste
> stable, ce qui est cohérent avec une prévention qui absorbe pour l'instant la
> hausse de l'aléa ». C'est plus juste, et bien plus solide en question.

## 6. Le LSTM — la question qui revient toujours

« Pour le temps, prends un LSTM » est le réflexe standard. Le projet l'a donc
construit, optimisé, et mesuré. Il perd. Cette section explique pourquoi, et
surtout pourquoi ce n'est pas un défaut d'implémentation.

### L'architecture, et le fait qu'elle a bien été réglée

Séquence de **30 jours × 8 indices CEMS** dans un LSTM, dont le dernier état
caché est concaténé aux 30 descripteurs de territoire et de calendrier, puis
passé dans une tête dense à deux couches.

L'objection « il n'a pas été fine-tuné » ne tient pas : **25 essais Optuna** sur
sept hyperparamètres, avec arrêt précoce.

In [ ]:
from tvfed.lstm import FENETRE, INDICES, STATIQUES

print(f"séquence   {FENETRE} jours × {len(INDICES)} indices = "
      f"{FENETRE * len(INDICES)} valeurs par ligne")
print(f"statiques  {len(STATIQUES)} descripteurs de territoire et de calendrier")
print(f"           AUCUNE feature dérivée de y — ni historique de feux, "
      f"ni taux lissé\n")
print("hyperparamètres retenus (25 essais Optuna, arrêt précoce) :")
for k, v in PARAMS.items():
    print(f"   {k:12s} {v}")

### La comparaison loyale n'est pas celle qu'on croit

XGBoost v3 voit l'historique des feux — dans le modèle A, la BDIFF pesait
**29 % des importances**. Le LSTM n'en voit rien. Les opposer mesurerait surtout
le prix de l'information retirée.

La seule référence à jeu d'information égal est le **modèle C** (physique pure,
41 features, 0 % dérivé de `y`). L'écart exact entre les deux :

| | modèle C | LSTM |
|---|---|---|
| territoire + calendrier | 30 features | **les mêmes 30** |
| météo | les 8 indices du jour + `danger_effis` + 2 décalages d'un jour | **30 jours × 8 indices = 240 valeurs** |

Le LSTM voit donc **vingt fois plus d'historique météo**. La seule chose qui lui
manque est `danger_effis` — et cette asymétrie joue contre lui, ce qui fait de
l'écart mesuré un **majorant**.

In [ ]:
'''FIG 5 — PR-AUC sur la validation, avec intervalles de confiance appariés.

Les intervalles viennent d'un bootstrap à 200 répliques rééchantillonnant les
34 734 COMMUNES, pas les lignes : les 1 096 jours d'une même commune ne sont
pas indépendants, et 31 communes partagent en moyenne la même maille météo. Un
bootstrap ligne à ligne produirait des intervalles faussement étroits.
'''
TAUX_BASE = 0.0002410
ap = PRAUC.iloc[0].to_dict()
ordre = sorted(ap, key=ap.get)
COUL = {"XGBoost v3": VERT, "DART": VERT, "MLP": VERT,
        "XGBoost C": BLEU, "LSTM": ROUGE}

fig, ax = plt.subplots(1, 2, figsize=(15.5, 4.6), width_ratios=[1, 1.15])

# ── (a) niveaux ──────────────────────────────────────────────────────────
ax[0].barh(range(len(ordre)), [ap[n] for n in ordre],
           color=[COUL[n] for n in ordre], height=.6)
ax[0].set_yticks(range(len(ordre)), ordre, fontsize=9.5)
ax[0].set_xlabel("PR-AUC sur la validation (38 M lignes, 9 176 feux)")
ax[0].set_title("(a)  Niveau de performance", fontsize=11.5, weight="bold",
                loc="left")
for i, n in enumerate(ordre):
    ax[0].text(ap[n] + .00035, i, f"{ap[n]:.4f}   {ap[n] / TAUX_BASE:.0f}×",
               va="center", fontsize=9, color=INK)
ax[0].set_xlim(0, max(ap.values()) * 1.38)
ax[0].set_ylim(-.95, len(ordre) - .45)
ax[0].axvline(TAUX_BASE, color=MUTED, ls=":", lw=1.2)
ax[0].annotate("hasard  (taux de base 0,024 %)", xy=(TAUX_BASE, -.55),
               xytext=(12, 0), textcoords="offset points", fontsize=8,
               color=MUTED, va="center")

# ── (b) écarts appariés ──────────────────────────────────────────────────
c = CMP[CMP.reference == "XGBoost v3"].copy()
c = pd.concat([c, CMP[(CMP.reference == "XGBoost C") & (CMP.modele == "LSTM")]])
c["lab"] = c.modele + "\nvs " + c.reference
c = c.iloc[::-1]
yy = np.arange(len(c))
for i, (_, r) in enumerate(c.iterrows()):
    coul = ROUGE if r.significatif else MUTED
    ax[1].plot([r.ic_bas, r.ic_haut], [i, i], color=coul, lw=2.4,
               solid_capstyle="round", zorder=2)
    ax[1].plot(r.ecart_pct, i, "o", color=coul, ms=6.5, zorder=3)
    ax[1].text(r.ic_haut + 1.6, i,
               f"{r.ecart_pct:+.1f} %" +
               ("" if r.significatif else "   (non significatif)"),
               va="center", fontsize=8.8, color=coul)
ax[1].axvline(0, color=INK, lw=1.1)
ax[1].set_yticks(yy, c.lab, fontsize=8.6)
ax[1].set_xlabel("écart de PR-AUC (%) — IC 95 % par bootstrap apparié")
ax[1].set_title("(b)  Les écarts survivent-ils au bruit ?", fontsize=11.5,
                weight="bold", loc="left")
ax[1].set_xlim(-72, 34)

for a in ax:
    a.grid(color=GRID, lw=.7, axis="x")
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False)
plt.show()

print(CMP.to_string(index=False,
                    columns=["reference", "modele", "ecart_pct", "ic_bas",
                             "ic_haut", "significatif"],
                    float_format=lambda v: f"{v:7.2f}"))

### Pourquoi il perd — l'explication est physique, pas informatique

Un LSTM sert quand **l'ordre de la séquence porte une information qu'aucun
résumé ne capture**. Ici, ce résumé existe déjà.

Les indices `DC`, `DMC` et `BUI` du système canadien **sont** des états
récursifs. Le *Drought Code* est littéralement une moyenne exponentielle de la
météo passée avec une constante de temps de **52 jours** ; le *Duff Moisture
Code*, de **15 jours**. Formellement :

$$\text{DC}_t = f(\text{DC}_{t-1},\ \text{pluie}_t,\ \text{température}_t)$$

C'est exactement la forme d'une cellule récurrente — sauf que ses coefficients
ont été calibrés par cinquante ans de science du feu plutôt qu'estimés sur
9 176 exemples positifs.

**Le CEMS livre déjà l'état caché.** Demander au LSTM de le ré-apprendre depuis
30 jours de séries brutes, c'est lui demander de redécouvrir une solution qu'on
lui donne en entrée sous forme fermée — avec un taux de base de 0,024 %, dans
un régime où chaque paramètre supplémentaire coûte cher.

Trois observations convergentes le confirment :

- la **PACF** (§3) montre une autocorrélation épuisée en **deux à trois jours** ;
- l'**ARIMA sans exogène** (§4) est inutilisable, r = −0,118 : le passé des feux
  ne prédit pas leur futur ;
- les trois premières features du modèle C sont `part_maquis` (26,2 %),
  `danger_effis` (13,7 %) et `erc` (11,1 %) — le signal dit **où il y a du
  combustible**, pas ce qui s'est passé le mois dernier.

Ce problème n'est pas une prévision de série temporelle. C'est une
**classification spatio-temporelle d'événement rare**, sur 34 734 séries
parallèles pilotées par un exogène déjà résumé par la physique du domaine. Le
réflexe « série temporelle → LSTM » vient des cours ; il ne survit pas au test.

## 7. Le bug qui a failli faire dire l'inverse

Le premier verdict annoncé pour le LSTM était **−97 %**. Le vrai est **−51,7 %**.
L'écart ne venait pas du modèle mais de la façon dont les deux fichiers de
prédictions étaient comparés — et le cas mérite d'être documenté, parce qu'il
est parfaitement silencieux.

`sql/50_matrice.sql` n'a **aucun `ORDER BY`**. L'ordre dans lequel PostgreSQL
renvoie les 38 M lignes dépend du plan d'exécution et des workers parallèles :
il **change d'une exécution à l'autre**. Les fichiers de prédictions ne
portaient que `(p, y)` — les comparer revenait à les aligner **par position**.

Deux fichiers issus de deux exécutions ont la même taille, le même nombre de
feux, et un ordre différent. **Rien ne signale l'erreur.**

In [ ]:
'''FIG 6 — Ce que coûte un désalignement d'une ligne sur deux.

On reproduit le bug volontairement : on permute les lignes d'un modèle, puis on
recalcule sa PR-AUC contre les cibles restées en place. La courbe précision-
rappel s'effondre sur la ligne du hasard, alors que le fichier contient
exactement les mêmes valeurs.
'''
from tvfed.comparer import ApRapide, aligner

A = aligner({"XGBoost v3": ("scores_val.parquet", "xgb_v3"),
             "LSTM": ("predictions_val_lstm.parquet", "p_lstm")}, bavard=False)
y = A.y.to_numpy(np.int8)
rng = np.random.default_rng(0)

droit = {n: ApRapide(A[n].to_numpy(), y)() for n in ("XGBoost v3", "LSTM")}
melange = rng.permutation(len(A))
casse = ApRapide(A.LSTM.to_numpy()[melange], y)()

print(f"{'':34s} {'PR-AUC':>8s}  {'lift':>7s}")
print(f"{'XGBoost v3 — aligné sur les clés':34s} {droit['XGBoost v3']:8.4f}  "
      f"{droit['XGBoost v3'] / TAUX_BASE:6.1f}×")
print(f"{'LSTM — aligné sur les clés':34s} {droit['LSTM']:8.4f}  "
      f"{droit['LSTM'] / TAUX_BASE:6.1f}×")
print(f"{'LSTM — lignes permutées':34s} {casse:8.4f}  {casse / TAUX_BASE:6.1f}×"
      f"   ← mêmes valeurs, autre ordre")

# ── où se trouvent les feux dans chaque classement ? ─────────────────────
fig, ax = plt.subplots(figsize=(9.2, 4.9))
seuils = np.array([.001, .002, .005, .01, .02, .05, .10, .20, .50])
for nom, p, coul, ls in (("XGBoost v3", A["XGBoost v3"].to_numpy(), VERT, "-"),
                         ("LSTM aligné", A.LSTM.to_numpy(), ROUGE, "-"),
                         ("LSTM permuté", A.LSTM.to_numpy()[melange], GRIS, "--")):
    o = np.argsort(-p)
    yo = y[o]
    part = [yo[:int(len(yo) * s)].sum() / y.sum() for s in seuils]
    ax.plot(seuils * 100, np.array(part) * 100, ls, color=coul, lw=1.9,
            marker="o", ms=4, label=nom)
ax.plot(seuils * 100, seuils * 100, ":", color=MUTED, lw=1.3, label="hasard")
ax.set_xscale("log")
ax.set_xlabel("part du territoire surveillée (% des communes-jours, échelle log)")
ax.set_ylabel("part des feux capturés (%)")
ax.set_title("Un désalignement ramène un modèle utile au niveau du hasard",
             fontsize=11.5, weight="bold", loc="left")
ax.legend(frameon=False, fontsize=9.5)
ax.grid(color=GRID, lw=.7)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### La parade retenue

Ajouter un `ORDER BY` coûterait un tri de 38 M lignes larges à chaque parcours.
Le projet a donc retenu l'autre solution :

1. **tout fichier de prédictions porte ses clés** `(code_insee, date)` ;
2. `tvfed.comparer.aligner()` trie sur ces clés et **vérifie** que les fichiers
   couvrent les mêmes lignes et portent les mêmes cibles ;
3. `tests/test_comparaison.py` **refuse** un fichier de prédictions sans clés.

Vérifié après coup : les fichiers `predictions_val_v1/v2/v3/dart/mlp`
partageaient bien le même ordre — par chance, pas par contrat. Les comparaisons
publiées entre ces modèles restent donc valides.

> **La leçon générale** : sur un événement à 0,024 %, une erreur de plomberie ne
> se manifeste jamais par une exception. Elle se manifeste par un chiffre
> plausible. Les seules défenses sont les invariants explicites et les
> assertions qui échouent bruyamment.

## Ce qu'il faut retenir

| # | Constat | Conséquence |
|---|---|---|
| 1 | Série stationnaire, autocorrélation épuisée en **2-3 jours** | pas de `d`, pas de dépendance longue à capter |
| 2 | L'ACF brute est illisible — le cycle annuel écrase tout | toujours retirer la saison avant de lire ACF/PACF |
| 2 bis | Le retard 8 est « significatif » à 0,034 (seuil 0,023) | à n = 7 305, significativité ≠ taille d'effet |
| 3 | ARIMA sans exogène : **r = −0,118** | le passé des feux ne prédit pas leur futur |
| 4 | Fourier + FWI : **−21,5 % de MAE** sur la référence saisonnière | la météo porte la prévisibilité |
| 5 | FWI estival **+ significatif sur 53 ans**, feux plats sur 20 ans | projeter l'aléa, jamais le bilan |
| 6 | LSTM **−23,6 %** [−33,5 ; −17,3] contre le modèle C | les indices CEMS sont déjà l'état récurrent |
| 7 | Un désalignement de lignes ramène un modèle au hasard | les prédictions portent leurs clés, un test le vérifie |

### Ce qui n'a pas été fait, et pourquoi

- **Un LSTM avec `danger_effis`** — le modèle C dispose de cette feature (13,7 %
  de son importance), pas le LSTM. L'écart de 23,6 % est donc un majorant. Vu
  que l'intervalle est loin de zéro et que `danger_effis` dérive du FWI, déjà
  présent dans la séquence, le classement ne devrait pas basculer — mais la
  question reste ouverte tant qu'elle n'est pas mesurée.
- **Un SARIMAX par région** — plus fin, mais le volet « combien » n'est pas le
  cœur du projet, et les effectifs régionaux annuels deviennent trop faibles.
- **Une évaluation sur 2023-2025** — le test reste vierge. Il ne sera ouvert
  qu'une fois, sur le modèle retenu.